# PIMA Neural Network Training

## Architecture Overview
This notebook implements a neural network for **PIMA Indians Diabetes Dataset - Binary classification**

### Configuration:
- **Dataset**: pima
- **Architecture**: 6 layers
- **Loss Function**: BCE
- **Optimizer**: Adam (lr=0.001)
- **Training**: 100 epochs, batch size 10

### Layer Structure:
- Layer 1: Linear with args [8, 10]
- Layer 2: ReLU
- Layer 3: Linear with args [10, 8]
- Layer 4: ReLU
- Layer 5: Linear with args [8, 1]
- Layer 6: Sigmoid


In [ ]:
# Install required packages
%pip install torch torchvision numpy scikit-learn matplotlib pandas

In [ ]:
# Import necessary libraries
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

In [ ]:
# Dataset Setup - PIMA Indians Diabetes (Tabular)
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

# Download and load PIMA dataset from URL
url = "https://raw.githubusercontent.com/npradaschnor/Pima-Indians-Diabetes-Dataset/master/diabetes.csv"
data = pd.read_csv(url)
X = data.iloc[:, :-1].values  # All columns except last
y = data.iloc[:, -1].values   # Last column (target)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Convert to tensors
import torch
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)

# Create datasets
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=10, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=10, shuffle=False)

print(f"PIMA Dataset loaded from URL:")
print(f"  Training samples: {len(train_dataset)}")
print(f"  Test samples: {len(test_dataset)}")
print(f"  Input features: {X_train.shape[1]}")
print(f"  Classes: Binary (0/1)")

In [ ]:
# Model Architecture for PIMA (Tabular)
import torch.nn as nn

class PimaModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Linear(8, 10),
            nn.ReLU(),
            nn.Linear(10, 8),
            nn.ReLU(),
            nn.Linear(8, 1),
            nn.Sigmoid()
        
        ])
    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

model = PimaModel()

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Model Architecture:")
print(model)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
model = model.to(device)
print(f"Model moved to: {device}")

In [ ]:
# Training Configuration for PIMA (Tabular)
import torch.optim as optim
import torch.nn as nn

epochs = 100
learning_rate = 0.001
batch_size = 10

loss_function = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

print("Training Configuration:")
print(f"  Epochs: {epochs}")
print(f"  Learning Rate: {learning_rate}")
print(f"  Loss Function: BCE")
print(f"  Optimizer: Adam")
print(f"  Batch Size: {batch_size}")

def train_epoch(model, train_loader, loss_fn, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = loss_fn(output, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        # For BCE, output is sigmoid, so threshold at 0.5
        preds = (output > 0.5).float()
        correct += (preds == target).sum().item()
        total += target.size(0)
    return total_loss / len(train_loader), 100. * correct / total

def evaluate(model, test_loader, loss_fn, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = loss_fn(output, target)
            total_loss += loss.item()
            preds = (output > 0.5).float()
            correct += (preds == target).sum().item()
            total += target.size(0)
    return total_loss / len(test_loader), 100. * correct / total

In [ ]:
# Training Loop for PIMA (Tabular)
import matplotlib.pyplot as plt

train_losses = []
train_accuracies = []
test_losses = []
test_accuracies = []

print("Starting training...")
print("=" * 50)

for epoch in range(1, 100 + 1):
    train_loss, train_acc = train_epoch(model, train_loader, loss_function, optimizer, device)
    test_loss, test_acc = evaluate(model, test_loader, loss_function, device)
    train_losses.append(train_loss)
    train_accuracies.append(train_acc)
    test_losses.append(test_loss)
    test_accuracies.append(test_acc)
    print(f'Epoch {epoch:2d}/100:')
    print(f'  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%')
    print(f'  Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%')
    print("-" * 50)

print("Training completed!")

final_train_acc = train_accuracies[-1]
final_test_acc = test_accuracies[-1]
overfitting = final_train_acc - final_test_acc

print("\nFinal Results:")
print(f"  Final Training Accuracy: {final_train_acc:.2f}%")
print(f"  Final Test Accuracy: {final_test_acc:.2f}%")
print(f"  Overfitting gap: {overfitting:.2f}%")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
ax1.plot(range(1, 100 + 1), train_losses, 'b-', label='Training Loss', linewidth=2)
ax1.plot(range(1, 100 + 1), test_losses, 'r-', label='Test Loss', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Test Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax2.plot(range(1, 100 + 1), train_accuracies, 'b-', label='Training Accuracy', linewidth=2)
ax2.plot(range(1, 100 + 1), test_accuracies, 'r-', label='Test Accuracy', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Training and Test Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
torch.save(model.state_dict(), 'trained_model.pth')
print("\nModel saved as 'trained_model.pth'")